In [15]:
!pip install selenium webdriver-manager


  Using cached typing_extensions-4.14.1-py3-none-any.whl.metadata (3.0 kB)
   ---------------------------------------- 0.0/9.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.4 MB 5.6 MB/s eta 0:00:02
   ------------------------ --------------- 5.8/9.4 MB 20.8 MB/s eta 0:00:01
   ---------------------------------- ----- 8.1/9.4 MB 16.2 MB/s eta 0:00:01
   ---------------------------------------- 9.4/9.4 MB 15.0 MB/s eta 0:00:00
Using cached typing_extensions-4.14.1-py3-none-any.whl (43 kB)
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.2.3
    Uninstalling urllib3-2.2.3:
      Successfully uninstalled urllib3-2.2.3
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.12.2
    Uninstalling typing_extensions-4.12.2:
      Successfully uninstalled typing_extensions-4.12.2
  Attempting uninstall: certifi
    Found existing installation: certifi 2024.8.30
    Uninstalling certifi-2024.8.30:
      Succes

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.2 requires numpy<2.1.0,>=2.0.0; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# DAAD Electrical Engineering Master's Scraper
Jupyter notebook to collect programme data (public universities, English-only) from DAAD.

**What it does:**
- Loads the filtered DAAD search results page (you paste its URL).
- Crawls all result pages and programme detail pages.
- Extracts: university, programme link, title/summary, IELTS/TOEFL, German req., min grade, tuition, city/state.
- Filters private universities (optional).
- Saves everything to Excel (`daad_electrical_eng_master.xlsx`).

**If DAAD blocks requests:** enable Selenium section (at the bottom).


In [3]:
# 1) Install dependencies (run once)
!pip install requests beautifulsoup4 lxml pandas openpyxl tqdm tenacity
# For Selenium fallback (optional):
# !pip install selenium webdriver-manager



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
# 2) Configuration

SEARCH_URL = "https://www2.daad.de/deutschland/studienangebote/international-programmes/en/result/?q=&degree%5B%5D=2&lang%5B%5D=2&fos=3&cert=&admReq=&langExamPC=&scholarshipLC=&langExamLC=&scholarshipSC=&langExamSC=&langDeAvailable=&langEnAvailable=&lvlEn%5B%5D=&modStd%5B%5D=&cit%5B%5D=&tyi%5B%5D=&ins%5B%5D=&fee=&bgn%5B%5D=&dat%5B%5D=&prep_subj%5B%5D=&prep_degree%5B%5D=&sort=4&dur=&subjects%5B%5D=&limit=10&offset=&display=list"  # <-- replace with your DAAD results URL
OUTPUT_XLSX = "daad_electrical_eng_master.xlsx"

ONLY_PUBLIC = True   # keep only public/state universities
USE_SELENIUM = True # set True if requests fail (see Selenium section)

# Request headers/timeouts
REQUESTS_HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}
TIMEOUT = 30
SLEEP_BETWEEN = (1, 2)  # seconds (min,max) between HTTP requests


In [19]:
# 3) Helper functions
import re, time, math, random
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlencode, urlparse, parse_qs
from tenacity import retry, stop_after_attempt, wait_exponential

BASE = "https://www2.daad.de"

def sleep():
    time.sleep(random.uniform(*SLEEP_BETWEEN))

def clean_text(txt):
    return re.sub(r"\s+", " ", txt).strip()

def extract_ielts_etc(text):
    ilt = re.search(r"IELTS[^0-9]*([\d\.]+)", text, re.I)
    toefl = re.search(r"TOEFL[^0-9]*([\d\.]+)", text, re.I)
    pte = re.search(r"PTE[^0-9]*([\d\.]+)", text, re.I)
    return {
        "IELTS": ilt.group(1) if ilt else None,
        "TOEFL": toefl.group(1) if toefl else None,
        "PTE": pte.group(1) if pte else None
    }

def german_req(text):
    if re.search(r"DSH|TestDaF|telc|Goethe", text, re.I):
        hits = set(re.findall(r"(DSH|TestDaF|telc|Goethe)", text, re.I))
        return "Yes (" + ", ".join(hits) + ")"
    return "None stated"

def extract_grade(text):
    # try to capture some common patterns
    m = re.search(r"(?:German|Overall)?\s*grade[^\d]*([1-5]\.?\d?)", text, re.I)
    if m:
        return clean_text(m.group(1))
    cgpa = re.search(r"CGPA[^\d]*([0-9]\.?[0-9]?)[/ ]*4", text, re.I)
    if cgpa:
        return f"CGPA {cgpa.group(1)}/4 (German equiv. not stated)"
    return "Not specified"

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=2, min=2, max=10))
def get(url):
    r = requests.get(url, headers=REQUESTS_HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r

def get_total_pages(soup):
    # Try to parse result count
    count_text = soup.select_one(".result-count, .search-result-count, .result__count")
    if count_text:
        m = re.search(r"(\d+)", count_text.get_text())
        if m:
            total = int(m.group(1))
            per_page = 10  # DAAD often shows 10 per page
            return math.ceil(total / per_page)
    # Fallback: look at pagination links
    pages = soup.select("ul.pagination li a")
    nums = [int(a.get_text()) for a in pages if a.get_text().isdigit()]
    return max(nums) if nums else 1

def parse_list_page(html):
    soup = BeautifulSoup(html, "lxml")
    cards = soup.select(".result-item, .course-list-item, .program-result-item, article")
    links = []
    for c in cards:
        a = c.select_one("a[href*='/international-programmes/en/detail/']") or c.select_one("a[href*='/detail/']")
        if a:
            links.append(urljoin(BASE, a['href']))
    # remove dupes, keep order
    return list(dict.fromkeys(links))

def parse_detail_page(html, link):
    soup = BeautifulSoup(html, "lxml")

    # Title / main subject
    title_el = soup.select_one("h1, .programme-title, h2")
    title = clean_text(title_el.get_text()) if title_el else ""

    # Uni name
    uni_el = soup.select_one(".university-name, .institution-name, .provider-name, h2.subtitle")
    uni = clean_text(uni_el.get_text()) if uni_el else ""

    # Course summary
    course_summary = ""
    summary_section = soup.find(lambda tag: tag.name in ["h3","h2"] and "Course" in tag.get_text())
    if summary_section:
        nxt = summary_section.find_next("p")
        if nxt:
            course_summary = clean_text(nxt.get_text())
    if not course_summary:
        desc = soup.select_one(".course-content, .programme-content, .description")
        if desc:
            course_summary = clean_text(desc.get_text())

    # Requirements text block
    req_block = soup.find(lambda tag: tag.name in ["h3","h2"] and re.search("Admission|Requirements|Language", tag.get_text(), re.I))
    req_text = ""
    if req_block:
        ptr = req_block
        while ptr and (ptr.name not in ["h2","h3"] or ptr == req_block):
            if ptr.name == "p":
                req_text += " " + ptr.get_text(" ")
            ptr = ptr.find_next_sibling()
    lang_text = (req_text or soup.get_text(" "))

    eng = extract_ielts_etc(lang_text)
    german = german_req(lang_text)
    grade = extract_grade(lang_text)

    # Tuition fees
    fee = "Not specified"
    fee_section = soup.find(lambda tag: tag.name in ["h3","h2"] and re.search("Tuition|Fees|Costs", tag.get_text(), re.I))
    if fee_section:
        ftxt = ""
        ptr = fee_section
        while ptr and (ptr.name not in ["h2","h3"] or ptr == fee_section):
            if ptr.name == "p":
                ftxt += " " + ptr.get_text(" ")
            ptr = ptr.find_next_sibling()
        if ftxt.strip():
            fee = clean_text(ftxt)

    # Location
    loc_el = soup.select_one(".location, .place, .city")
    location = clean_text(loc_el.get_text()) if loc_el else ""
    city, state = "", ""
    if "," in location:
        city, state = [s.strip() for s in location.split(",", 1)]
    else:
        city = location

    # University type (public/private)
    uni_type = ""
    type_cell = soup.find(text=re.compile("University type", re.I))
    if type_cell and type_cell.parent:
        nxt = type_cell.parent.find_next("td")
        if nxt:
            uni_type = clean_text(nxt.get_text())

    # Official programme link
    official_link = ""
    for a in soup.select("a[href^='http']"):
        if "website" in a.get_text().lower() or "university" in a.get_text().lower():
            official_link = a['href']
            break

    return {
        "University": uni,
        "Program Link (DAAD)": link,
        "Official Programme URL": official_link,
        "Program Title/Main Subject": title,
        "Course Summary": course_summary[:500],
        "English Req.": f"IELTS: {eng['IELTS'] or 'n/a'}, TOEFL: {eng['TOEFL'] or 'n/a'}, PTE: {eng['PTE'] or 'n/a'}",
        "German Req.": german,
        "Min Grade (German/CGPA note)": grade,
        "Tuition Fee": fee,
        "City": city,
        "State": state,
        "Uni Type": uni_type
    }


In [13]:
# 4) Scrape with requests (default path)

from tqdm import tqdm

def scrape_requests():
    first = get(SEARCH_URL)
    soup = BeautifulSoup(first.text, "lxml")
    total_pages = get_total_pages(soup)
    print(f"Detected {total_pages} pages.")

    all_links = parse_list_page(first.text)

    parsed = urlparse(SEARCH_URL)
    qs = parse_qs(parsed.query)

    for page in range(2, total_pages + 1):
        qs['page'] = [str(page)]
        page_url = parsed._replace(query=urlencode(qs, doseq=True)).geturl()
        sleep()
        rp = get(page_url)
        all_links.extend(parse_list_page(rp.text))

    all_links = list(dict.fromkeys(all_links))
    print(f"Total programme links: {len(all_links)}")    

    rows = []
    for link in tqdm(all_links, desc="Programmes"):
        sleep()
        try:
            r = get(link)
            data = parse_detail_page(r.text, link)
            rows.append(data)
        except Exception as e:
            rows.append({
                "University": "ERROR",
                "Program Link (DAAD)": link,
                "Official Programme URL": "",
                "Program Title/Main Subject": str(e),
                "Course Summary": "",
                "English Req.": "",
                "German Req.": "",
                "Min Grade (German/CGPA note)": "",
                "Tuition Fee": "",
                "City": "",
                "State": "",
                "Uni Type": ""
            })
    return rows

def drop_private(df):
    if "Uni Type" in df.columns:
        return df[~df["Uni Type"].str.contains("private", case=False, na=False)].copy()
    return df

rows = scrape_requests()
df = pd.DataFrame(rows)

if ONLY_PUBLIC:
    df = drop_private(df)

# Reorder columns
cols = [
    "University",
    "Program Link (DAAD)",
    "Official Programme URL",
    "Program Title/Main Subject",
    "Course Summary",
    "English Req.",
    "German Req.",
    "Min Grade (German/CGPA note)",
    "Tuition Fee",
    "City",
    "State"
]
df = df.reindex(columns=cols)

df.to_excel(OUTPUT_XLSX, index=False)
print(f"Saved -> {OUTPUT_XLSX} (Rows: {len(df)})")
df.head()


Detected 1 pages.
Total programme links: 0


Programmes: 0it [00:00, ?it/s]

Saved -> daad_electrical_eng_master.xlsx (Rows: 0)


,University,Program Link (DAAD),Official Programme URL,Program Title/Main Subject,Course Summary,English Req.,German Req.,Min Grade (German/CGPA note),Tuition Fee,City,State


## 5) Selenium fallback (only if requests fail)
If DAAD blocks or returns empty content, enable Selenium. Uncomment install cell above and run the next cell.
Set `USE_SELENIUM = True` and adjust the code below.

In [21]:
# OPTIONAL: Selenium fallback
# Uncomment & adapt if needed. Only run if requests path fails.

if USE_SELENIUM:
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    import bs4

    def get_selenium(url):
        driver.get(url)
        WebDriverWait(driver, 20).until(lambda d: d.execute_script('return document.readyState') == 'complete')
        return driver.page_source

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

    html = get_selenium(SEARCH_URL)
    soup = BeautifulSoup(html, "lxml")
    total_pages = get_total_pages(soup)
    all_links = parse_list_page(html)

    parsed = urlparse(SEARCH_URL)
    qs = parse_qs(parsed.query)
    for page in range(2, total_pages + 1):
        qs['page'] = [str(page)]
        page_url = parsed._replace(query=urlencode(qs, doseq=True)).geturl()
        html = get_selenium(page_url)
        all_links.extend(parse_list_page(html))

    all_links = list(dict.fromkeys(all_links))
    rows = []
    for link in tqdm(all_links, desc="Programmes (Selenium)"):
        html = get_selenium(link)
        data = parse_detail_page(html, link)
        rows.append(data)

    driver.quit()

    df = pd.DataFrame(rows)
    if ONLY_PUBLIC:
        df = drop_private(df)
    df = df.reindex(columns=cols)
    df.to_excel(OUTPUT_XLSX, index=False)
    print(f"Saved -> {OUTPUT_XLSX} (Rows: {len(df)})")
    df.head()


Programmes (Selenium): 0it [00:00, ?it/s]


Saved -> daad_electrical_eng_master.xlsx (Rows: 0)
